In [ ]:
import requests
from config import CONTACT_EMAIL
url = "https://data.sec.gov/api/xbrl/companyfacts/CIK0000100493.json"

headers = {
    "User-Agent": f"ArkansasFinancialAnalysis/1.0 contact: {CONTACT_EMAIL}"
}

response = requests.get(url, headers=headers)

print(response.status_code)

200


In [52]:
data = response.json()

print(data.keys())
print(data["entityName"])
print(data["facts"].keys())

facts = data["facts"]["us-gaap"]

dict_keys(['cik', 'entityName', 'facts'])
TYSON FOODS, INC.
dict_keys(['dei', 'us-gaap', 'ffd'])


In [53]:
##Revenue Extraction

revenue = facts["Revenues"]["units"]["USD"]

tyson_revenue = []

for year in range(2020,2026):
    yearly_items = []

    for item in revenue:
        if(
            item["form"] == "10-K"
            and item["fp"] == "FY"
            and item ["fy"] == year
        ):
            yearly_items.append(item)

    latest_item = max(yearly_items, key=lambda x: x["end"])

    tyson_revenue.append({
        "fiscal_year": year,
        "revenue":latest_item["val"]
    })

tyson_revenue

[{'fiscal_year': 2020, 'revenue': 43185000000},
 {'fiscal_year': 2021, 'revenue': 47049000000},
 {'fiscal_year': 2022, 'revenue': 53282000000},
 {'fiscal_year': 2023, 'revenue': 52881000000},
 {'fiscal_year': 2024, 'revenue': 53309000000},
 {'fiscal_year': 2025, 'revenue': 54441000000}]

In [54]:
## Operating income extraction. Tyson works differently in that the filter I've been using returned multiple observations for the same fiscal year,
## including quarterly periods. Using the duration of the reporting period was not reliable enough on its own. I noticed the full-year observation
## has the latest end date for that discal year, so for each year, I collected the observation and used max() with lambda to select the observation
## with the latest end date, This returned the complete dictionary associated with the latest reporting period.

operating_income = facts["OperatingIncomeLoss"]["units"]["USD"]

tyson_operating_income = []

for year in range(2020, 2026):
    
    yearly_items = []

    for item in operating_income:
        if (
            item["form"] == "10-K"
            and item["fp"] == "FY"
            and item["fy"] == year
        ):
            yearly_items.append(item)

    latest_item = max(yearly_items, key=lambda x: x["end"])

## as you can see I am grabbing each item in each year, and finding the full year observation by checking which is tha latest.
    
    tyson_operating_income.append({
        "fiscal_year": year,
        "operating_income": latest_item["val"]
    })

tyson_operating_income

[{'fiscal_year': 2020, 'operating_income': 3114000000},
 {'fiscal_year': 2021, 'operating_income': 4396000000},
 {'fiscal_year': 2022, 'operating_income': 4410000000},
 {'fiscal_year': 2023, 'operating_income': -395000000},
 {'fiscal_year': 2024, 'operating_income': 1409000000},
 {'fiscal_year': 2025, 'operating_income': 1098000000}]

In [55]:
net_income = facts["NetIncomeLoss"]["units"]["USD"]

tyson_net_income = []

for year in range(2020, 2026):
    yearly_items = []

    for item in net_income:
        if(
            item["form"] == "10-K"
            and item["fp"] == "FY"
            and item["fy"] == year
        ):
            yearly_items.append(item)
    latest_item  = max(yearly_items, key = lambda x: x["end"])

    tyson_net_income.append({
    "fiscal_year": year,
    "net_income": latest_item["val"]
    })

tyson_net_income

[{'fiscal_year': 2020, 'net_income': 2140000000},
 {'fiscal_year': 2021, 'net_income': 3047000000},
 {'fiscal_year': 2022, 'net_income': 3238000000},
 {'fiscal_year': 2023, 'net_income': -648000000},
 {'fiscal_year': 2024, 'net_income': 800000000},
 {'fiscal_year': 2025, 'net_income': 474000000}]

In [56]:
assets = facts["Assets"]["units"]["USD"]

tyson_assets = []

for year in range(2020, 2026):
    yearly_items = []

    for item in assets:
        if (
            item["form"] == "10-K"
            and item["fp"] == "FY"
            and item["fy"] == year
        ):
            yearly_items.append(item)

    latest_item = max(yearly_items, key=lambda x: x["end"])

    tyson_assets.append({
        "fiscal_year": year,
        "total_assets": latest_item["val"]
    })

tyson_assets

[{'fiscal_year': 2020, 'total_assets': 34741000000},
 {'fiscal_year': 2021, 'total_assets': 36309000000},
 {'fiscal_year': 2022, 'total_assets': 36821000000},
 {'fiscal_year': 2023, 'total_assets': 36251000000},
 {'fiscal_year': 2024, 'total_assets': 37100000000},
 {'fiscal_year': 2025, 'total_assets': 36658000000}]

In [57]:
equity = facts["StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"]["units"]["USD"]

tyson_equity = []

for year in range(2020, 2026):
    yearly_items = []

    for item in equity:
        if (
            item["form"] == "10-K"
            and item["fp"] == "FY"
            and item["fy"] == year
        ):
            yearly_items.append(item)

    latest_item = max(yearly_items, key=lambda x: x["end"])

    tyson_equity.append({
        "fiscal_year": year,
        "total_equity": latest_item["val"]
    })

tyson_equity

[{'fiscal_year': 2020, 'total_equity': 15597000000},
 {'fiscal_year': 2021, 'total_equity': 17854000000},
 {'fiscal_year': 2022, 'total_equity': 19811000000},
 {'fiscal_year': 2023, 'total_equity': 18255000000},
 {'fiscal_year': 2024, 'total_equity': 18514000000},
 {'fiscal_year': 2025, 'total_equity': 18227000000}]

In [58]:
assets = facts["Assets"]["units"]["USD"]
equity = facts["StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"]["units"]["USD"]

liabilities_data = []

for year in range(2020, 2026):

    yearly_assets = []
    yearly_equity = []

    for item in assets:
        if (
            item["form"] == "10-K"
            and item["fp"] == "FY"
            and item["fy"] == year
        ):
            yearly_assets.append(item)

    for item in equity:
        if (
            item["form"] == "10-K"
            and item["fp"] == "FY"
            and item["fy"] == year
        ):
            yearly_equity.append(item)

    latest_asset = max(yearly_assets, key=lambda x: x["end"])
    latest_equity = max(yearly_equity, key=lambda x: x["end"])

    liabilities = latest_asset["val"] - latest_equity["val"]

    liabilities_data.append({
        "fiscal_year": year,
        "total_liabilities": liabilities
    })

print(liabilities_data)

[{'fiscal_year': 2020, 'total_liabilities': 19144000000}, {'fiscal_year': 2021, 'total_liabilities': 18455000000}, {'fiscal_year': 2022, 'total_liabilities': 17010000000}, {'fiscal_year': 2023, 'total_liabilities': 17996000000}, {'fiscal_year': 2024, 'total_liabilities': 18586000000}, {'fiscal_year': 2025, 'total_liabilities': 18431000000}]


In [59]:
operating_cash_flow = facts["NetCashProvidedByUsedInOperatingActivities"]["units"]["USD"]

tyson_operating_cash_flow = []

for year in range(2020, 2026):
    yearly_items = []

    for item in operating_cash_flow:
        if (
            item["form"] == "10-K"
            and item["fp"] == "FY"
            and item["fy"] == year
        ):
            yearly_items.append(item)

    latest_item = max(yearly_items, key=lambda x: x["end"])

    tyson_operating_cash_flow.append({
        "fiscal_year": year,
        "operating_cash_flow": latest_item["val"]
    })

tyson_operating_cash_flow

[{'fiscal_year': 2020, 'operating_cash_flow': 3874000000},
 {'fiscal_year': 2021, 'operating_cash_flow': 3840000000},
 {'fiscal_year': 2022, 'operating_cash_flow': 2687000000},
 {'fiscal_year': 2023, 'operating_cash_flow': 1752000000},
 {'fiscal_year': 2024, 'operating_cash_flow': 2590000000},
 {'fiscal_year': 2025, 'operating_cash_flow': 2155000000}]

In [60]:
import pandas as pd

other_data = [
    tyson_operating_income,
    tyson_net_income,
    tyson_assets,
    tyson_equity,
    liabilities_data,
    tyson_operating_cash_flow
]
              
    
tyson_df = pd.DataFrame(tyson_revenue)

for data in other_data:
    temp_df = pd.DataFrame(data)
    tyson_df = tyson_df.merge(temp_df, on="fiscal_year")

In [61]:
tyson_df["company"] = "Tyson"

In [63]:
tyson_df.to_csv("tyson_financial_data.csv", index =False)